# Detekcija vatre i dima na slikama (Skup podataka: D-Fire)
**Predmet:** Prepoznavanje oblika i obrada slike (ETF Sarajevo, 2026/2027)  
**Tim:** Vatrogasci (Danijal Alibegović, Aldin Velić, Zijad Doglod, Dino Burić, Haris Macić, Adnan Šemić)  

---

### Pregled strukture notebook-a (Koraci 5–10 iz plana projekta)
Prva četiri koraka (istraživanje teme, analiza radova, profiliranje D-Fire skupa i definisanje problema) dokumentovana su u glavnom projektnom izvještaju. Ovaj notebook pokriva praktičnu realizaciju:
1. **Korak 5:** Učitavanje skupa podataka, provjera strukture i preprocesiranje (stratifikovana podjela na `train`/`val`/`test`)
2. **Korak 6:** Konfiguracija izabranih modela i pristupa (YOLOv8 baseline + poređenje)
3. **Korak 7:** Implementacija i treniranje modela
4. **Korak 8:** Evaluacija pristupa, analiza grešaka i fine-tuning parametara
5. **Korak 9:** Demonstracija / priprema modela za korisnički interfejs i izvoz rezultata za dokumentaciju

## 0. Podešavanje okruženja i reproducibilnost

In [2]:
# Instalacija potrebnih biblioteka (otkomentarisati po potrebi, npr. na Colab/Kaggle)
# !pip install -q ultralytics opencv-python-headless scikit-learn matplotlib seaborn pyyaml

import os
import random
import shutil
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import ultralytics
from ultralytics import YOLO

# Fiksni seed za reproducibilnost svih eksperimenata
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch verzija: {torch.__version__} | CUDA dostupna: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU uređaj: {torch.cuda.get_device_name(0)}")
print(f"Ultralytics verzija: {ultralytics.__version__}")

PyTorch verzija: 2.15.0+cpu | CUDA dostupna: False
Ultralytics verzija: 8.4.215


---
## 1. Učitavanje skupa podataka i preprocesiranje

**Ciljevi ovog dijela (prema profiliranju D-Fire skupa):**
* Učitavanje Kaggle distribucije skupa D-Fire (`21.527` slika, klase: `0: smoke` / `1: fire` — provjeriti tačan redoslijed u `data.yaml`!).
* Provjera uparenosti slika (`.jpg`) i oznaka (`.txt`), uključujući prazne `.txt` datoteke za negativne slike (`9.838` slika).
* **Stratifikovana podjela:** Izvorni `train` skup (`17.221` slika) nema odvojen validacijski skup, pa ga dijelimo na `train` i `val` (npr. 80/20 od trening skupa, odnosno ukupno ~64% train, ~16% val, 20% test) uz očuvanje omjera sve 4 kategorije slika (*samo vatra*, *samo dim*, *vatra i dim*, *bez vatre i dima*).

In [6]:
import hashlib
from collections import Counter
from pathlib import Path
from PIL import Image
import pandas as pd

# Podesi samo ovu putanju ako ti se folder zove drugačije (npr. "./data" ili "./Smoke-Fire-Detection-YOLO")
DATASET_DIR = Path("./data")

def provjeri_dfire_samostalno(korijen: Path):
    if not korijen.exists():
        raise FileNotFoundError(f"Folder '{korijen.resolve()}' ne postoji! Provjeri putanju DATASET_DIR.")

    slike = []
    hashovi = {"train": Counter(), "test": Counter()}

    for skup in ["train", "test"]:
        img_dir = korijen / skup / "images"
        lbl_dir = korijen / skup / "labels"

        if not img_dir.exists() or not lbl_dir.exists():
            raise FileNotFoundError(f"Nedostaju folderi '{img_dir}' ili '{lbl_dir}'.")

        for putanja in sorted(img_dir.glob("*.jpg")):
            with Image.open(putanja) as slika:
                slika.verify()
                w, h = slika.size

            oznaka = lbl_dir / f"{putanja.stem}.txt"
            if not oznaka.exists():
                raise FileNotFoundError(f"Nedostaje label fajl za sliku: {putanja.name}")

            klase = [
                int(r.split()[0])
                for r in oznaka.read_text(encoding="utf-8-sig").splitlines()
                if r.strip()
            ]
            if any(k not in (0, 1) for k in klase):
                raise ValueError(f"Nepoznata klasa u {oznaka}")

            dim, vatra = klase.count(0), klase.count(1)
            if dim and vatra:
                kategorija = "vatra_i_dim"
            elif dim:
                kategorija = "samo_dim"
            elif vatra:
                kategorija = "samo_vatra"
            else:
                kategorija = "bez_vatre_i_dima"

            h_val = hashlib.sha256(putanja.read_bytes()).hexdigest()
            hashovi[skup][h_val] += 1

            slike.append({
                "skup": skup,
                "sirina": w,
                "visina": h,
                "dim": dim,
                "vatra": vatra,
                "kategorija": kategorija
            })

    df = pd.DataFrame(slike)
    duplikati = len(hashovi["train"].keys() & hashovi["test"].keys())
    print(f"OK! Ukupno slika: {len(df)} | Zajednički SHA-256 između train i test: {duplikati}")
    print("\nRaspodjela slika po kategorijama:")
    print(pd.crosstab(df["kategorija"], df["skup"], margins=True))

provjeri_dfire_samostalno(DATASET_DIR)

OK! Ukupno slika: 18428 | Zajednički SHA-256 između train i test: 0

Raspodjela slika po kategorijama:
skup              test  train    All
kategorija                          
bez_vatre_i_dima  2005   6458   8463
samo_dim          1186   3836   5022
samo_vatra         220    770    990
vatra_i_dim        895   3058   3953
All               4306  14122  18428


In [7]:
import shutil
from pathlib import Path
import pandas as pd
import yaml
from sklearn.model_selection import train_test_split

# Podesi putanje i parametre direktno ovdje
DATASET_DIR = Path("./data")
WORK_DIR = Path("./dfire_prepared")
VAL_RATIO = 0.2
SEED = 42

def pripremi_dfire_samostalno(korijen: Path, work_dir: Path, val_ratio: float = 0.2, seed: int = 42):
    if not korijen.exists():
        raise FileNotFoundError(f"Folder '{korijen.resolve()}' ne postoji! Provjeri putanju DATASET_DIR.")

    zapisi = []
    for skup in ["train", "test"]:
        img_dir = korijen / skup / "images"
        lbl_dir = korijen / skup / "labels"

        for putanja in sorted(img_dir.glob("*.jpg")):
            oznaka = lbl_dir / f"{putanja.stem}.txt"
            klase = [
                int(r.split()[0])
                for r in oznaka.read_text(encoding="utf-8-sig").splitlines()
                if r.strip()
            ]
            dim, vatra = klase.count(0), klase.count(1)
            if dim and vatra:
                kategorija = "vatra_i_dim"
            elif dim:
                kategorija = "samo_dim"
            elif vatra:
                kategorija = "samo_vatra"
            else:
                kategorija = "bez_vatre_i_dima"

            zapisi.append({
                "skup": skup,
                "putanja_slike": putanja,
                "putanja_oznake": oznaka,
                "kategorija": kategorija
            })

    df = pd.DataFrame(zapisi)
    df_train_full = df[df["skup"] == "train"].copy()
    df_test = df[df["skup"] == "test"].copy()

    df_train, df_val = train_test_split(
        df_train_full,
        test_size=val_ratio,
        random_state=seed,
        stratify=df_train_full["kategorija"]
    )

    podskupovi = {"train": df_train, "val": df_val, "test": df_test}

    for naziv, pod_df in podskupovi.items():
        img_out = work_dir / naziv / "images"
        lbl_out = work_dir / naziv / "labels"
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)

        for _, red in pod_df.iterrows():
            shutil.copy2(red["putanja_slike"], img_out / red["putanja_slike"].name)
            shutil.copy2(red["putanja_oznake"], lbl_out / red["putanja_oznake"].name)

        print(f"{naziv.upper():5s}: {len(pod_df):5d} slika | {dict(pod_df['kategorija'].value_counts())}")

    data_yaml = {
        "path": str(work_dir.resolve()),
        "train": "train/images",
        "val": "val/images",
        "test": "test/images",
        "nc": 2,
        "names": {0: "smoke", 1: "fire"}
    }
    yaml_putanja = work_dir / "data.yaml"
    with open(yaml_putanja, "w", encoding="utf-8") as f:
        yaml.dump(data_yaml, f, sort_keys=False)

    print(f"\nGotovo! Sačuvan fajl: {yaml_putanja.resolve()}")

pripremi_dfire_samostalno(DATASET_DIR, WORK_DIR, val_ratio=VAL_RATIO, seed=SEED)

TRAIN: 11297 slika | {'bez_vatre_i_dima': np.int64(5166), 'samo_dim': np.int64(3069), 'vatra_i_dim': np.int64(2446), 'samo_vatra': np.int64(616)}
VAL  :  2825 slika | {'bez_vatre_i_dima': np.int64(1292), 'samo_dim': np.int64(767), 'vatra_i_dim': np.int64(612), 'samo_vatra': np.int64(154)}
TEST :  4306 slika | {'bez_vatre_i_dima': np.int64(2005), 'samo_dim': np.int64(1186), 'vatra_i_dim': np.int64(895), 'samo_vatra': np.int64(220)}

Gotovo! Sačuvan fajl: C:\Users\Alibegovici\Desktop\ETF\Master\Prva\POOS\Projekat\izvjestaj\dfire_prepared\data.yaml


---
## 2. Opis i konfiguracija izabranih modela

Na osnovu analize literature (Poglavlje 1) i definisanog pristupa (Poglavlje 2), testiramo sljedeće konfiguracije pod jednakim uslovima (`imgsz=640`, isti `train/val/test` split):
1. **Baseline model:** `YOLOv8n` (ili `YOLOv8s`) sa podrazumijevanim hiperparametrima.
2. **Srednja / naprednija varijanta:** `YOLOv8m` ili `YOLO11s` (poređenje arhitektura).
3. **Poboljšani model (fine-tuning / modifikacija):** prilagođena augmentacija (npr. mozaik, HSV pomaci) i/ili podešavanje funkcije gubitka / praga pouzdanosti radi smanjenja lažnih alarma na negativnim slikama.

In [ ]:
# Zajednički hiperparametri za poređenje konfiguracija
CONFIG = {
    "data_yaml": str(WORK_DIR / "data.yaml"),
    "imgsz": 640,
    "epochs": 100,
    "batch": 32,          # prilagoditi dostupnoj VRAM memoriji (npr. 16 ili 32)
    "patience": 15,       # rano zaustavljanje (early stopping)
    "seed": SEED,
    "project": "runs_dfire"
}

print("Osnovna konfiguracija eksperimenta:", CONFIG)

---
## 3. Implementacija i treniranje modela

### 3.1 Eksperiment 1 — Baseline model (YOLOv8n / YOLOv8s)

In [ ]:
# Učitavanje unaprijed treniranih težina i pokretanje treninga
# model_base = YOLO("yolov8n.pt")
# results_base = model_base.train(
#     data=CONFIG["data_yaml"],
#     epochs=CONFIG["epochs"],
#     imgsz=CONFIG["imgsz"],
#     batch=CONFIG["batch"],
#     patience=CONFIG["patience"],
#     seed=CONFIG["seed"],
#     project=CONFIG["project"],
#     name="exp1_yolov8n_baseline"
# )

### 3.2 Eksperiment 2 — Druga arhitektura / veći model (npr. YOLO11s ili YOLOv8m)

In [ ]:
# TODO: Treniranje drugog modela pod istim uslovima radi poređenja

### 3.3 Eksperiment 3 — Poboljšani model (prilagođene augmentacije / hiperparametri)

In [ ]:
# TODO: Treniranje konfiguracije sa podešenim augmentacijama / hiperparametrima

---
## 4. Evaluacija pristupa, analiza grešaka i fine-tuning

Prema definisanim ciljevima projekta, evaluacija na `test` skupu uključuje:
1. **Detekcijske metrike:** `mAP@0.5`, `mAP@0.5:0.95`, `Precision` i `Recall` ukupno i **zasebno za klase `fire` i `smoke`**.
2. **Klasifikacija prisutnosti na nivou slike:** Mapiranje detekcija u 4 kategorije (*samo vatra*, *samo dim*, *vatra i dim*, *bez pojava*) i iscrtavanje **matrice konfuzije**.
3. **Stopa lažnih alarma (FPR):** Procenat pogrešnih detekcija na negativnim slikama (magla, oblaci, odsjaj).
4. **Kvalitativna analiza grešaka:** Vizuelni prikaz primjera sa lažno pozitivnim i propuštenim detekcijama (mali plamenovi, noćne scene).

In [ ]:
# TODO: Evaluacija najboljih težina (best.pt) na test skupu
# best_model = YOLO("runs_dfire/exp1_yolov8n_baseline/weights/best.pt")
# metrics = best_model.val(data=CONFIG["data_yaml"], split="test")

# Ovdje dodati:
# - Tabelarni pregled metrika po modelima (za direktno prebacivanje u LaTeX tabele)
# - Evaluaciju na nivou cijele slike (4 kategorije) i matricu konfuzije
# - Prikaz karakterističnih grešaka (FP i FN)

---
## 5. Priprema za korisnički interfejs i demo inferencija

Jednostavna funkcija za testiranje istreniranog modela na proizvoljnoj slici, sa ispisom detektovanih klasa (`fire`, `smoke`), kategorije slike i prikazom graničnih okvira.

In [ ]:
def predict_fire_smoke(model, image_path, conf_threshold=0.25):
    """
    Pokreće inferenciju nad jednom slikom i vraća kategoriju prisutnosti
    (Samo vatra / Samo dim / Vatra i dim / Bez vatre i dima) uz vizuelni prikaz.
    """
    results = model.predict(source=str(image_path), conf=conf_threshold, verbose=False)[0]
    detected_classes = set(results.names[int(cls_id)] for cls_id in results.boxes.cls)
    
    has_fire = "fire" in detected_classes
    has_smoke = "smoke" in detected_classes
    
    if has_fire and has_smoke:
        category = "Vatra i dim"
    elif has_fire:
        category = "Samo vatra"
    elif has_smoke:
        category = "Samo dim"
    else:
        category = "Bez vatre i dima"
        
    print(f"Slika: {Path(image_path).name} -> Kategorija: {category} (detektovano objekata: {len(results.boxes)})")
    
    # Prikaz slike sa graničnim okvirima
    annotated = results.plot()[:, :, ::-1]  # BGR u RGB
    plt.figure(figsize=(8, 6))
    plt.imshow(annotated)
    plt.title(f"Rezultat detekcije: {category}")
    plt.axis("off")
    plt.show()
    return category, results